# Notebook 2 - Clasificacion y Grad-CAM (BUSI)

**Deep Learning en Python para el Modelado Predictivo** - Trabajo academico

**Autores:** _(rellenar)_  ·  **Entrega:** 24 de mayo de 2026

---

Clasificacion de ecografias de mama en 3 clases (**benigno / maligno / normal**) con
**transferencia del conocimiento**, comparando **VGG16 y ResNet50** y analizando la
explicabilidad con **Grad-CAM**.

**Indice**
1. Carga de los datos
2. Analisis exploratorio (EDA)
3. Particion 80/20 y preprocesado
4. Data augmentation
5. Modelo: transferencia del conocimiento y capas congeladas
6. Validacion cruzada (8 combinaciones x 4 folds)
7. Entrenamiento del modelo final
8. Evaluacion en test
9. Comparacion VGG16 vs ResNet50
10. Grad-CAM (explicabilidad)
11. Grad-CAM vs mascara
12. Conclusiones

Pensado para ejecutarse en un servidor con GPU.

In [ ]:
# Montaje de Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, re, time, gc, random

# ── Reproducibilidad (fijar ANTES de importar TensorFlow) ────────────────────
os.environ['PYTHONHASHSEED']         = '42'
os.environ['TF_DETERMINISTIC_OPS']   = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import cv2

import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras import Model, layers
from tensorflow.keras.applications import VGG16, ResNet50, imagenet_utils
from tensorflow.keras.layers import Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, f1_score, roc_auc_score)

# ── Memory growth: TF solo reserva VRAM cuando la necesita (evita OOM) ───────
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# ----------------- CONFIGURACION (AJUSTA si tu carpeta esta en otra ruta) -----------------
BASE            = '/content/drive/MyDrive/Deep Learning'
DATA_DIR        = BASE + '/Dataset_BUSI_clean'         # salida del notebook 1 (imagenes validas)
DESCARTADAS_DIR = BASE + '/Dataset_BUSI_descartadas'   # salida del notebook 1 (imagenes eliminadas)
OUT_DIR         = BASE + '/resultados'                 # resultados (compartidos con el notebook 3)

CLASSES   = ['benign', 'malignant', 'normal']
N_CLASSES = 3
IMG_SIZE  = (224, 224)
BATCH_SIZE = 32
SEED = 42

# --- Hiperparametros que optimiza la validacion cruzada -----------------------
# Rejilla = lr x dropout x n_bloques  ->  2 x 2 x 2 = 8 combinaciones
LEARNING_RATES = [1e-4, 1e-5]   # learning rate
DROPOUTS       = [0.0, 0.5]     # tasa de Dropout en la 1a capa Dense del clasificador
N_BLOQUES      = [0, 1]         # 0 = solo clasificador (base congelada) | 1 = + ultimo bloque conv
N_FOLDS        = 4              # folds de la validacion cruzada
MODELOS        = ['VGG16', 'ResNet50']

CAPA_GRADCAM = {'VGG16': 'block5_conv3', 'ResNet50': 'conv5_block3_out'}

# --- Entrenamiento ------------------------------------------------------------
CV_MAX_EPOCHS = 30   # epocas maximas por fold en la validacion cruzada
CV_PATIENCE   = 5    # paciencia del EarlyStopping en cada fold
# El nº de epocas del entrenamiento final NO es fijo: es la media de las epocas
# utiles (epocas_totales - paciencia) de los 4 folds de la combinacion ganadora.

os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED)            # Python built-in
np.random.seed(SEED)         # NumPy
tf.random.set_seed(SEED)     # TensorFlow / inicializacion de pesos Dense
assert os.path.isdir(DATA_DIR), 'No se encuentra DATA_DIR: ' + DATA_DIR
print('DATA_DIR =', DATA_DIR)

## 1. Carga de los datos

Cargamos todas las imagenes limpias en memoria, en un array `X` y un vector `y`,
igual que la P4 cargaba CIFAR10. Guardamos ademas la ruta y la mascara de cada imagen
(haran falta para el Grad-CAM y para el notebook 3 de segmentacion).

In [ ]:
def cargar_dataset(data_dir):
    X, y, rutas, mascaras = [], [], [], []
    for idx, cls in enumerate(CLASSES):
        cdir = os.path.join(data_dir, cls)
        ficheros = sorted(os.listdir(cdir))
        imagenes = [f for f in ficheros if f.lower().endswith('.png') and '_mask' not in f.lower()]
        for f in imagenes:
            img = Image.open(os.path.join(cdir, f)).convert('RGB').resize(IMG_SIZE)
            X.append(np.array(img, dtype='uint8'))   # uint8 ahorra ~300 MB de RAM
            y.append(idx)
            rutas.append(os.path.join(cdir, f))
            stem = f[:-4]
            mascaras.append([os.path.join(cdir, m) for m in ficheros if m.startswith(stem + '_mask')])
    return np.array(X), np.array(y), rutas, mascaras

t0 = time.time()
X, y, rutas, mascaras = cargar_dataset(DATA_DIR)
print('[TIEMPO] Carga de imagenes: %.1f s' % (time.time() - t0))
print('Imagenes cargadas:', X.shape, X.dtype, '(%d MB)' % (X.nbytes // (1024 * 1024)))
for i, c in enumerate(CLASSES):
    print('  ', c, '->', int((y == i).sum()))

## 2. Analisis exploratorio (EDA)

In [ ]:
conteo = pd.Series(y).value_counts().sort_index()
ax = conteo.plot.bar(color='seagreen', figsize=(6, 4))
ax.set_xticklabels(CLASSES, rotation=0); ax.set_ylabel('nº de imagenes')
ax.set_title('Distribucion de clases (dataset limpio)')
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom')
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'distribucion.png'), dpi=120); plt.show()
print('Dataset DESBALANCEADO -> usaremos class_weight y la metrica F1-macro.')

In [ ]:
# Una imagen de cada clase con su mascara
plt.figure(figsize=(13, 8))
for i, cls in enumerate(CLASSES):
    pos = np.where(y == i)[0][0]
    plt.subplot(2, 3, i + 1)
    plt.imshow(X[pos].astype('uint8')); plt.axis('off'); plt.title(cls)
    m = np.zeros(IMG_SIZE, np.uint8)
    for mp in mascaras[pos]:
        mm = np.array(Image.open(mp).convert('L').resize(IMG_SIZE))
        m = np.maximum(m, (mm > 127).astype(np.uint8))
    plt.subplot(2, 3, i + 4)
    plt.imshow(X[pos].astype('uint8')); plt.imshow(m, alpha=0.4, cmap='Reds')
    plt.axis('off'); plt.title(cls + ' + mascara')
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'muestras.png'), dpi=120); plt.show()

In [ ]:
# Ejemplos de imagenes ELIMINADAS en la limpieza (figura para el paper)
if os.path.isdir(DESCARTADAS_DIR):
    rng = np.random.RandomState(SEED)
    plt.figure(figsize=(11, 11))
    for i, cls in enumerate(CLASSES):
        cdir = os.path.join(DESCARTADAS_DIR, cls)
        imgs = sorted([f for f in os.listdir(cdir)
                       if f.lower().endswith('.png') and '_mask' not in f.lower()])
        sel = rng.choice(len(imgs), size=min(3, len(imgs)), replace=False)
        for j, s in enumerate(sel):
            ax = plt.subplot(3, 3, i * 3 + j + 1)
            ax.imshow(Image.open(os.path.join(cdir, imgs[s])).convert('RGB'))
            ax.axis('off')
            ax.set_title(cls + ' - eliminada', fontsize=10)
    plt.suptitle('Ejemplos de imagenes eliminadas en la limpieza (3 por clase)',
                 fontsize=13)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, 'descartadas_ejemplos.png'),
                dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('[AVISO] No se encuentra', DESCARTADAS_DIR,
          '- se omite descartadas_ejemplos.png')

## 3. Particion 80/20 y preprocesado

Particion **estratificada** train (80 %) / test (20 %). El test se aparta y no se usa
hasta la evaluacion final. **No se reserva un conjunto de validacion fijo**: los
hiperparametros se eligen mediante validacion cruzada sobre el train (seccion 6).

Preprocesado con `imagenet_utils.preprocess_input` (valido para VGG16 y ResNet50) y
etiquetas a one-hot. Como el dataset esta desbalanceado calculamos aqui el
`class_weight` que usaran tanto la validacion cruzada como el entrenamiento final.

Se guardan los indices del split (`split_indices.npz`) para que el **notebook 3 de
U-Net** use exactamente el mismo conjunto de test.

In [ ]:
# Particion estratificada 80% train / 20% test (sin conjunto de validacion fijo)
indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.20,
                                       stratify=y, random_state=SEED)

# X esta en uint8 (RAM); aqui casteamos a float32 antes del preprocess
Xtrain = imagenet_utils.preprocess_input(X[idx_train].astype('float32'))
Xtest  = imagenet_utils.preprocess_input(X[idx_test].astype('float32'))

ytrain_int, ytest_int = y[idx_train], y[idx_test]
ytrain = to_categorical(ytrain_int, N_CLASSES)
ytest  = to_categorical(ytest_int, N_CLASSES)

# Dataset desbalanceado -> class_weight (lo usan la CV y el entrenamiento final)
counts_por_clase = np.array([int((ytrain_int == i).sum()) for i in range(N_CLASSES)])
ratio_desbalance = counts_por_clase.max() / max(counts_por_clase.min(), 1)
if ratio_desbalance > 1.5:
    pesos = compute_class_weight('balanced', classes=np.arange(N_CLASSES), y=ytrain_int)
    class_weight = dict(enumerate(pesos))
    print('Dataset DESBALANCEADO (ratio %.2f) -> class_weight:' % ratio_desbalance,
          {CLASSES[i]: round(w, 2) for i, w in class_weight.items()})
else:
    class_weight = None
    print('Clases BALANCEADAS -> class_weight = None')

# Guardamos el split para que el notebook 3 (U-Net) use el mismo conjunto de test
np.savez(os.path.join(OUT_DIR, 'split_indices.npz'),
         idx_train=idx_train, idx_test=idx_test)

print('train:', len(idx_train), '| test:', len(idx_test))
print('Imagenes por clase en train:', dict(zip(CLASSES, counts_por_clase.tolist())))
print('Xtrain en RAM: %d MB' % (Xtrain.nbytes // (1024 * 1024)))

## 4. Data augmentation

Generamos variaciones sinteticas de las imagenes de entrenamiento (volteos,
rotaciones, zoom y desplazamientos) para mitigar el overfitting. Usamos **capas de
preprocesado de Keras** dentro de un pipeline `tf.data`: a diferencia de
`ImageDataGenerator`, no acumulan memoria al entrenar muchos modelos en bucle
(la validacion cruzada entrena decenas de modelos seguidos).

In [ ]:
# Data augmentation con capas de preprocesado de Keras.
# Se usa tf.data.from_tensor_slices (NO from_generator): asi NO se acumula memoria
# al entrenar muchos modelos en bucle, que es lo que bloqueaba la sesion de Colab.
augmentador = tf.keras.Sequential([
    layers.RandomFlip('horizontal', seed=SEED),
    layers.RandomRotation(15.0 / 360.0, fill_mode='nearest', seed=SEED),  # +/-15 grados
    layers.RandomZoom(0.1, fill_mode='nearest', seed=SEED),
    layers.RandomTranslation(0.1, 0.1, fill_mode='nearest', seed=SEED),
], name='augmentador')

def make_train_ds(X, y, batch_size):
    """tf.data.Dataset de entrenamiento con augmentation (sin fuga de memoria)."""
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    ds = ds.shuffle(len(X), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size)
    ds = ds.map(lambda xb, yb: (augmentador(xb, training=True), yb),
                num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

print('Augmentation (capas de Keras) lista.')

## 5. Modelo: transferencia del conocimiento y capas congeladas

Cargamos la red base (**VGG16** o **ResNet50**) con pesos de ImageNet y sin
clasificador (`include_top=False`).

**Capas congeladas (fine-tuning).**
- `0` → **transfer learning puro**: base 100 % congelada, solo se entrena el clasificador.
- `1` → se descongela el **ultimo bloque** convolucional + clasificador.

Las capas `BatchNormalization` se mantienen **siempre** congeladas, incluso al
descongelar bloques convolucionales, para preservar el *running mean* y la
*running variance* estimados durante el preentrenamiento en ImageNet.

**Clasificador propio:**
`Flatten → Dense(256, relu) → [Dropout(p)] → Dense(128, relu) → Dense(3, softmax)`.

El Dropout se aplica **solo entre las capas Dense** del clasificador, nunca en la base
convolucional. Su tasa `p` no se fija a priori: es uno de los hiperparametros que
optimiza la validacion cruzada (`p ∈ {0.0, 0.5}`).

In [ ]:
PREFIJOS = {'VGG16':    ['block5', 'block4'],
            'ResNet50': ['conv5',  'conv4']}

def crear_modelo(nombre, learning_rate=1e-4, n_bloques=1, dropout_rate=0.0):
    """Crea VGG16/ResNet50 preentrenada en ImageNet + clasificador propio.

    learning_rate -> tasa de aprendizaje de Adam.
    n_bloques     -> 0 = base congelada | 1 = ultimo bloque conv descongelado.
    dropout_rate  -> tasa de Dropout entre las capas Dense del clasificador.
                     Es un hiperparametro de la validacion cruzada (0.0 o 0.5).
    """
    if nombre == 'VGG16':
        base = VGG16(weights='imagenet', include_top=False, input_shape=IMG_SIZE + (3,))
    else:
        base = ResNet50(weights='imagenet', include_top=False, input_shape=IMG_SIZE + (3,))

    descongelar = PREFIJOS[nombre][:n_bloques]
    for layer in base.layers:
        layer.trainable = any(layer.name.startswith(p) for p in descongelar)
    for layer in base.layers:
        if isinstance(layer, BatchNormalization):
            layer.trainable = False

    last = base.layers[-1].output
    x = Flatten()(last)
    x = Dense(256, activation='relu', name='fc1')(x)
    if dropout_rate > 0:
        x = Dropout(dropout_rate)(x)
    x = Dense(128, activation='relu', name='fc2')(x)
    x = Dense(N_CLASSES, activation='softmax', name='predictions')(x)
    modelo = Model(base.input, x)
    modelo.compile(optimizer=Adam(learning_rate),
                   loss='categorical_crossentropy', metrics=['accuracy'])
    return modelo

crear_modelo('VGG16', n_bloques=1, dropout_rate=0.5).summary()

## 6. Validacion cruzada (8 combinaciones x 4 folds)

Para **cada modelo** se optimizan **3 hiperparametros** con validacion cruzada
estratificada (`StratifiedKFold`, k=4) **sobre el conjunto de train**:

- `learning_rate` ∈ {1e-4, 1e-5}
- `dropout`       ∈ {0.0, 0.5}  → tasa de Dropout en el clasificador
- `n_bloques`     ∈ {0, 1}      → 0 = solo clasificador | 1 = + ultimo bloque conv

Son 2 × 2 × 2 = **8 combinaciones**. Cada combinacion se entrena en los **4 folds**
con data augmentation, `class_weight` y **EarlyStopping** (`val_loss`, paciencia=5,
maximo 30 epocas). Para cada fold se registra:

- el **F1-macro** sobre el fold de validacion (metrica de seleccion, porque las clases
  estan desbalanceadas: promedia el F1 de cada clase por igual);
- las **epocas utiles** = epocas totales entrenadas − paciencia, que corresponden a
  la epoca de mejor `val_loss`.

Se reportan la **media** y la **desviacion tipica** de ambas magnitudes sobre los 4
folds. La combinacion ganadora es la de mayor F1-macro medio; su media de epocas
utiles fija la duracion del entrenamiento final (seccion 7).

> La CV usa augmentation y `class_weight` igual que el entrenamiento final, para que la
> estimacion del numero de epocas sea trasladable al modelo definitivo.
>
> **La CV es reanudable y se apoya en los CSV.** `validacion_cruzada()` guarda el
> progreso en `cv_<modelo>.csv` tras cada combinacion y, al ejecutarse, **lee ese CSV**:
> si ya esta completo lo reutiliza sin reentrenar; si esta a medias continua donde se
> quedo. Asi, si la sesion de Colab se bloquea, basta con reiniciar y volver a ejecutar
> el notebook entero: lo ya calculado se carga del disco, sin depender de las variables
> en memoria.

In [ ]:
def validacion_cruzada(nombre):
    """Validacion cruzada estratificada (k=N_FOLDS) sobre el conjunto de train.

    8 combinaciones (lr x dropout x n_bloques). REANUDABLE y basada en CSV:
      - si 'cv_<nombre>.csv' ya esta completo  -> lo CARGA y devuelve (no reentrena);
      - si esta a medias                       -> continua donde se quedo;
      - si no existe                           -> calcula desde cero.
    Guarda el CSV tras CADA combinacion, asi que un bloqueo de Colab no pierde
    progreso: basta reiniciar y volver a ejecutar el notebook.
    """
    ruta_csv = os.path.join(OUT_DIR, 'cv_' + nombre + '.csv')
    combos = [(lr, dr, nb) for lr in LEARNING_RATES
                            for dr in DROPOUTS
                            for nb in N_BLOQUES]

    # --- Reanudacion: cargar lo ya calculado en una ejecucion anterior ---
    if os.path.exists(ruta_csv):
        filas = pd.read_csv(ruta_csv).to_dict('records')
        if len(filas) >= len(combos):
            print('CV de %s YA COMPLETA -> se carga %s (sin reentrenar).'
                  % (nombre, os.path.basename(ruta_csv)))
            return (pd.DataFrame(filas)
                    .sort_values('f1_macro_mean', ascending=False)
                    .reset_index(drop=True))
        print('CV de %s REANUDADA: %d de %d combinaciones ya hechas.'
              % (nombre, len(filas), len(combos)))
    else:
        filas = []
    n_previos = len(filas)

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    print('=== Validacion cruzada:', nombre, '(', len(combos),
          'combinaciones x', N_FOLDS, 'folds ) ===')
    t_total = time.time()
    for k, (lr, dr, nb) in enumerate(combos, start=1):
        if k <= n_previos:               # combinacion ya hecha en una sesion previa
            continue
        t_combo = time.time()
        f1s, epocas = [], []
        for tr_i, va_i in skf.split(Xtrain, ytrain_int):
            # --- limpieza de memoria ANTES de crear el modelo ---
            K.clear_session()
            gc.collect()

            modelo = crear_modelo(nombre, lr, nb, dropout_rate=dr)
            early = EarlyStopping(monitor='val_loss', patience=CV_PATIENCE,
                                  restore_best_weights=True, verbose=0)
            ds_tr = make_train_ds(Xtrain[tr_i], ytrain[tr_i], BATCH_SIZE)
            H = modelo.fit(ds_tr,
                           validation_data=(Xtrain[va_i], ytrain[va_i]),
                           epochs=CV_MAX_EPOCHS, class_weight=class_weight,
                           callbacks=[early], verbose=0)
            pred = modelo.predict(Xtrain[va_i], verbose=0,
                                  batch_size=BATCH_SIZE).argmax(axis=1)
            f1s.append(f1_score(ytrain_int[va_i], pred, average='macro'))
            # epocas utiles = epocas totales entrenadas - paciencia (mejor val_loss)
            epocas.append(max(1, len(H.history['loss']) - CV_PATIENCE))

            # --- limpieza de memoria DESPUES del fit/predict ---
            del modelo, pred, H, ds_tr
            gc.collect()
        filas.append({'lr': lr, 'dropout': dr, 'n_bloques': nb,
                      'f1_macro_mean': float(np.mean(f1s)),
                      'f1_macro_std':  float(np.std(f1s)),
                      'epochs_mean':   float(np.mean(epocas)),
                      'epochs_std':    float(np.std(epocas))})
        # --- Guardado incremental: sobrevive a un bloqueo de la sesion ---
        pd.DataFrame(filas).to_csv(ruta_csv, index=False)
        hechos = k - n_previos
        eta = (time.time() - t_total) / hechos * (len(combos) - k)
        print('  [%d/%d] lr=%.0e drop=%.1f nb=%d -> F1=%.3f±%.3f | epocas=%.1f±%.1f'
              '  | combo %.1fmin | ETA %.1fmin' %
              (k, len(combos), lr, dr, nb,
               filas[-1]['f1_macro_mean'], filas[-1]['f1_macro_std'],
               filas[-1]['epochs_mean'],   filas[-1]['epochs_std'],
               (time.time() - t_combo) / 60, eta / 60))
    print('=== %s: CV completa -> %s ===' % (nombre, os.path.basename(ruta_csv)))
    return (pd.DataFrame(filas)
            .sort_values('f1_macro_mean', ascending=False)
            .reset_index(drop=True))

In [ ]:
# CV de VGG16. La funcion guarda/lee cv_VGG16.csv sola:
# si ya existe completo lo carga en 1 segundo; si no, lo calcula (o reanuda).
cv_vgg = validacion_cruzada('VGG16')
cv_vgg

In [ ]:
# CV de ResNet50. La funcion guarda/lee cv_ResNet50.csv sola:
# si ya existe completo lo carga; si no (o esta a medias) lo calcula/reanuda.
cv_resnet = validacion_cruzada('ResNet50')
cv_resnet

In [ ]:
# A partir de aqui el notebook se apoya SOLO en los ficheros guardados en resultados/.
# Releemos las dos tablas de CV directamente desde sus CSV (no de variables en memoria).
def cargar_cv(nombre):
    """Lee cv_<nombre>.csv y lo devuelve ordenado por F1-macro (mejor primero)."""
    ruta = os.path.join(OUT_DIR, 'cv_' + nombre + '.csv')
    return (pd.read_csv(ruta)
            .sort_values('f1_macro_mean', ascending=False)
            .reset_index(drop=True))

cv_vgg    = cargar_cv('VGG16')
cv_resnet = cargar_cv('ResNet50')

# Visualizacion: F1-macro de CADA combinacion (8 barras por modelo) con su desv. tipica
plt.style.use('ggplot')
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for k, (nombre, tabla) in enumerate([('VGG16', cv_vgg), ('ResNet50', cv_resnet)]):
    ax = axes[k]
    # Orden fijo (lr, dropout, n_bloques) para que sea comparable entre modelos
    t = tabla.sort_values(['lr', 'dropout', 'n_bloques'],
                          ascending=[False, True, True]).reset_index(drop=True)
    etiquetas = ['lr=%.0e\ndrop=%.1f\nnb=%d'
                 % (r['lr'], r['dropout'], int(r['n_bloques']))
                 for _, r in t.iterrows()]
    best_idx = int(t['f1_macro_mean'].idxmax())
    colores = ['seagreen' if i == best_idx else 'steelblue' for i in range(len(t))]
    ax.bar(range(len(t)), t['f1_macro_mean'].values,
           yerr=t['f1_macro_std'].values, color=colores, capsize=4,
           error_kw={'elinewidth': 1.2, 'ecolor': '#333333'})
    for i, r in t.iterrows():
        ax.text(i, r['f1_macro_mean'] + r['f1_macro_std'] + 0.03,
                '%.3f' % r['f1_macro_mean'], ha='center', fontsize=8)
    ax.set_xticks(range(len(t)))
    ax.set_xticklabels(etiquetas, fontsize=8)
    ax.set_title(nombre + ' - F1-macro por combinacion (4-fold CV, verde = mejor)')
    ax.set_ylabel('F1-macro'); ax.set_ylim(0, 1.08)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'cv_combos.png'), dpi=120, bbox_inches='tight')
plt.show()

# La combinacion ganadora = primera fila del CSV ordenado por F1-macro.
# Se LEE del fichero, asi el entrenamiento final NO depende de variables en memoria.
mejor = {n: cargar_cv(n).iloc[0] for n in MODELOS}
for nombre in MODELOS:
    b = mejor[nombre]
    print('Mejor %-9s -> lr=%.0e drop=%.1f n_bloques=%d  |  '
          'F1-macro=%.3f±%.3f  |  epocas=%.1f±%.1f'
          % (nombre, b['lr'], b['dropout'], int(b['n_bloques']),
             b['f1_macro_mean'], b['f1_macro_std'],
             b['epochs_mean'], b['epochs_std']))

## 7. Entrenamiento del modelo final

Para cada modelo se entrena **un unico modelo definitivo** con la **combinacion
ganadora** de la validacion cruzada (mayor F1-macro medio): su `learning_rate`, su
`dropout` y su `n_bloques`. Esa combinacion se **lee del fichero** `cv_<modelo>.csv`,
de modo que esta seccion no depende de las variables en memoria.

El numero de epocas **no es fijo ni usa EarlyStopping**: es la **media de las epocas
utiles** de los 4 folds de esa combinacion (redondeada al entero mas proximo). Asi la
duracion del entrenamiento la decide la propia validacion cruzada.

El entrenamiento usa **todo el 80 % de train**, data augmentation y `class_weight`, y
guarda el modelo en `modelo_<modelo>.keras` y su historial en `hist_<modelo>.csv`.
**Si ese `.keras` ya existe, el modelo se carga en vez de reentrenarse**: asi, si la
sesion de Colab se reinicia entre el entrenamiento de VGG16 y el de ResNet50, basta
con volver a ejecutar el notebook entero. Las curvas muestran solo `train_loss` y
`train_accuracy` (no hay conjunto de validacion).

In [ ]:
def entrenar_final(nombre):
    """Entrena (o CARGA, si ya existe) el modelo definitivo con la combinacion
    ganadora de la CV. Deja en disco:
      - modelo_<nombre>.keras  -> el modelo entrenado
      - hist_<nombre>.csv      -> historial de entrenamiento (para las curvas)
    Si modelo_<nombre>.keras ya existe, se CARGA y NO se reentrena (util si la
    sesion de Colab se reinicia entre VGG16 y ResNet50). Para forzar el
    reentrenamiento, borra ese .keras de la carpeta resultados/.

    El nº de epocas = MEDIA de las epocas utiles de los 4 folds REDONDEADA HACIA
    ARRIBA (np.ceil), para no quedarnos cortos por el redondeo."""
    ruta_modelo = os.path.join(OUT_DIR, 'modelo_' + nombre + '.keras')
    ruta_hist   = os.path.join(OUT_DIR, 'hist_'   + nombre + '.csv')

    if os.path.exists(ruta_modelo):
        print('Modelo final %s YA EXISTE -> se carga %s (sin reentrenar).'
              % (nombre, os.path.basename(ruta_modelo)))
        modelo = tf.keras.models.load_model(ruta_modelo)
        history = (pd.read_csv(ruta_hist).to_dict('list')
                   if os.path.exists(ruta_hist) else {'loss': [], 'accuracy': []})
        return modelo, history

    b = mejor[nombre]
    epocas = max(1, int(np.ceil(b['epochs_mean'])))
    print('--- Entrenamiento final %s: lr=%.0e drop=%.1f n_bloques=%d -> %d epocas '
          '(media CV = %.1f, redondeada hacia arriba) ---'
          % (nombre, b['lr'], b['dropout'], int(b['n_bloques']), epocas, b['epochs_mean']))
    modelo = crear_modelo(nombre, float(b['lr']), int(b['n_bloques']),
                          dropout_rate=float(b['dropout']))
    H = modelo.fit(make_train_ds(Xtrain, ytrain, BATCH_SIZE),
                   epochs=epocas, class_weight=class_weight, verbose=1)
    modelo.save(ruta_modelo)
    pd.DataFrame(H.history).to_csv(ruta_hist, index=False)   # historial para las curvas
    print('Guardado:', os.path.basename(ruta_modelo), '+', os.path.basename(ruta_hist))
    return modelo, H.history

def curvas(history, titulo):
    """Curvas del entrenamiento final (solo train, sin conjunto de validacion)."""
    if not history or len(history.get('loss', [])) == 0:
        print('(%s: modelo cargado de disco sin historial -> se omiten las curvas)'
              % titulo)
        return
    plt.style.use('ggplot')
    plt.figure(figsize=(13, 4))
    ep = np.arange(len(history['loss']))
    plt.subplot(1, 2, 1)
    plt.plot(ep, history['loss'], marker='o', color='steelblue', label='train_loss')
    plt.title(titulo + ' - Loss'); plt.xlabel('Epoch #'); plt.ylabel('loss'); plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(ep, history['accuracy'], marker='o', color='seagreen', label='train_acc')
    plt.title(titulo + ' - Accuracy'); plt.xlabel('Epoch #'); plt.ylabel('accuracy'); plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, 'curvas_' + titulo.replace(' ', '_') + '.png'), dpi=120)
    plt.show()

In [ ]:
# Entrenamiento del modelo definitivo VGG16
t0 = time.time()
modelo_vgg, hist_vgg = entrenar_final('VGG16')
print('[TIEMPO] VGG16 final: %.1f min  (%d epocas)' %
      ((time.time() - t0) / 60, len(hist_vgg['loss'])))
curvas(hist_vgg, 'VGG16')

In [ ]:
# Entrenamiento del modelo definitivo ResNet50
t0 = time.time()
modelo_resnet, hist_resnet = entrenar_final('ResNet50')
print('[TIEMPO] ResNet50 final: %.1f min  (%d epocas)' %
      ((time.time() - t0) / 60, len(hist_resnet['loss'])))
curvas(hist_resnet, 'ResNet50')

# Los modelos definitivos que se usaran en evaluacion, comparativa y Grad-CAM
modelos = {'VGG16': modelo_vgg, 'ResNet50': modelo_resnet}
print('Modelos definitivos listos para evaluacion.')

## 8. Evaluacion en el conjunto de test

Metricas adecuadas a un problema desbalanceado y medico: classification report
(precision, recall, F1 por clase y macro), matriz de confusion y AUC.

In [ ]:
def evaluar(nombre, modelo):
    prob = modelo.predict(Xtest, verbose=0)
    pred = prob.argmax(axis=1)
    print('=============== ' + nombre + ' ===============')
    print(classification_report(ytest_int, pred, target_names=CLASSES, digits=3))
    cm = confusion_matrix(ytest_int, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASSES)
    disp.plot(cmap='Blues', values_format='d')
    disp.ax_.grid(False)            # quita la rejilla de ggplot (cruz blanca sobre los numeros)
    disp.ax_.set_title('Matriz de confusion - ' + nombre)
    plt.savefig(os.path.join(OUT_DIR, 'cm_' + nombre + '.png'),
                dpi=120, bbox_inches='tight')
    plt.show()
    acc = float((pred == ytest_int).mean())
    f1  = f1_score(ytest_int, pred, average='macro')
    auc = roc_auc_score(ytest_int, prob, multi_class='ovr', average='macro')
    print('Accuracy=%.3f | F1-macro=%.3f | AUC=%.3f\n' % (acc, f1, auc))
    return {'modelo': nombre, 'accuracy': acc, 'f1_macro': f1, 'auc': auc}

res_vgg    = evaluar('VGG16', modelo_vgg)
res_resnet = evaluar('ResNet50', modelo_resnet)

## 9. Comparacion VGG16 vs ResNet50

In [ ]:
comparativa = pd.DataFrame([res_vgg, res_resnet]).set_index('modelo')
print(comparativa.round(3))
comparativa.plot.bar(figsize=(8, 4), rot=0)
plt.title('Comparacion de modelos (test)'); plt.ylabel('valor'); plt.ylim(0, 1)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'comparativa.png'), dpi=120); plt.show()

comparativa.to_csv(os.path.join(OUT_DIR, 'test_metrics.csv'))   # accuracy, F1-macro, AUC

mejor_modelo_nombre = comparativa['f1_macro'].idxmax()
print('Mejor modelo segun F1-macro:', mejor_modelo_nombre)

## 10. Grad-CAM (explicabilidad)

**Grad-CAM** genera un **mapa de calor** que indica en que zona de la imagen se ha
fijado la red para decidir la clase (rojo = mas influyente). Usa los gradientes de la
ultima capa convolucional.

In [ ]:
def grad_cam(img_rgb, modelo, capa):
    x = imagenet_utils.preprocess_input(np.expand_dims(img_rgb.astype('float32'), axis=0).copy())
    grad_model = Model(modelo.input, [modelo.output, modelo.get_layer(capa).output])
    with tf.GradientTape() as tape:
        pred, conv_out = grad_model(x)
        clase = tf.argmax(pred[0])
        salida_clase = pred[:, clase]
    grads  = tape.gradient(salida_clase, conv_out)
    pooled = K.mean(grads, axis=(0, 1, 2))
    heatmap = tf.reduce_mean(tf.multiply(pooled, conv_out), axis=-1)[0].numpy()
    heatmap = np.maximum(heatmap, 0)
    heatmap = heatmap / (heatmap.max() + 1e-8)
    return cv2.resize(heatmap, IMG_SIZE), int(clase)

def superponer(heatmap, img_rgb, intensidad=0.4):
    hm = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)[:, :, ::-1]
    return np.uint8((1 - intensidad) * img_rgb + intensidad * hm)

In [ ]:
# Grad-CAM de los dos modelos sobre las mismas imagenes de test (1 por clase)
modelos = {'VGG16': modelo_vgg, 'ResNet50': modelo_resnet}
plt.figure(figsize=(13, 9))
for i, cls in enumerate(CLASSES):
    p = idx_test[ytest_int == i][0]
    img = X[p].astype('uint8')
    plt.subplot(3, 3, i * 3 + 1)
    plt.imshow(img); plt.axis('off'); plt.title('real: ' + cls, fontsize=9)
    for k, nombre in enumerate(MODELOS):
        heatmap, clase = grad_cam(X[p], modelos[nombre], CAPA_GRADCAM[nombre])
        plt.subplot(3, 3, i * 3 + 2 + k)
        plt.imshow(superponer(heatmap, img)); plt.axis('off')
        plt.title(nombre + '  pred: ' + CLASSES[clase], fontsize=9)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'gradcam_ejemplos.png'), dpi=120); plt.show()

## 11. Grad-CAM vs mascara

Para cada imagen **con tumor** del test comparamos el Grad-CAM con la **mascara real**:
- **inside_frac**: fraccion del calor dentro del tumor.
- **IoU** y **DICE**: solapamiento del mapa binarizado con la mascara.
- **acierto_pico**: el punto mas caliente cae dentro del tumor?

Los resultados se guardan en `localizacion.csv` para que el **notebook 3** los compare
con los de la U-Net.

In [ ]:
def cargar_mascara(lista_mascaras):
    m = np.zeros(IMG_SIZE, np.uint8)
    for mp in lista_mascaras:
        mm = np.array(Image.open(mp).convert('L').resize(IMG_SIZE, Image.NEAREST))
        m = np.maximum(m, (mm > 127).astype(np.uint8))
    return m

def analizar_localizacion(nombre, modelo):
    capa = CAPA_GRADCAM[nombre]
    filas = []
    for p in idx_test:
        if CLASSES[y[p]] == 'normal':
            continue
        mask = cargar_mascara(mascaras[p])
        if mask.sum() == 0:
            continue
        heatmap, _ = grad_cam(X[p], modelo, capa)
        hb = (heatmap >= 0.5).astype('float32')
        inter = float((hb * mask).sum()); union = float(((hb + mask) > 0).sum())
        pico = np.unravel_index(int(np.argmax(heatmap)), heatmap.shape)
        filas.append({'modelo': nombre, 'clase': CLASSES[y[p]],
                      'inside_frac': float((heatmap * mask).sum() / (heatmap.sum() + 1e-8)),
                      'IoU': inter / (union + 1e-8),
                      'DICE': 2 * inter / (hb.sum() + mask.sum() + 1e-8),
                      'acierto_pico': bool(mask[pico] > 0)})
    return pd.DataFrame(filas)

loc_vgg    = analizar_localizacion('VGG16', modelo_vgg)
loc_resnet = analizar_localizacion('ResNet50', modelo_resnet)
loc = pd.concat([loc_vgg, loc_resnet], ignore_index=True)
loc.to_csv(os.path.join(OUT_DIR, 'localizacion.csv'), index=False)
print('Imagenes con tumor evaluadas por modelo:', len(loc_vgg))

In [ ]:
resumen_loc = loc.groupby('modelo')[['inside_frac', 'IoU', 'DICE', 'acierto_pico']].mean()
print(resumen_loc.round(3))
resumen_loc.plot.bar(figsize=(8, 4), rot=0)
plt.title('Localizacion Grad-CAM vs mascara'); plt.ylabel('valor medio')
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'localizacion.png'), dpi=120); plt.show()

In [ ]:
# Ejemplos visuales del mejor modelo: imagen | mascara real | Grad-CAM
modelo_top = modelos[mejor_modelo_nombre]
capa_top   = CAPA_GRADCAM[mejor_modelo_nombre]
loc_top = loc[loc['modelo'] == mejor_modelo_nombre].reset_index(drop=True)
tumor_idx = [p for p in idx_test if CLASSES[y[p]] != 'normal' and cargar_mascara(mascaras[p]).sum() > 0]
orden = loc_top['inside_frac'].sort_values()
sel = list(orden.index[:3]) + list(orden.index[-3:])

plt.figure(figsize=(11, 14))
for fila, k in enumerate(sel):
    p = tumor_idx[k]
    img = X[p].astype('uint8')
    heatmap, _ = grad_cam(X[p], modelo_top, capa_top)
    mask = cargar_mascara(mascaras[p])
    plt.subplot(6, 3, fila * 3 + 1); plt.imshow(img); plt.axis('off'); plt.title(CLASSES[y[p]], fontsize=9)
    plt.subplot(6, 3, fila * 3 + 2); plt.imshow(img); plt.imshow(mask, alpha=0.45, cmap='Greens')
    plt.axis('off'); plt.title('mascara real', fontsize=9)
    plt.subplot(6, 3, fila * 3 + 3); plt.imshow(superponer(heatmap, img)); plt.axis('off')
    plt.title('Grad-CAM (in=%.2f)' % loc_top.loc[k, 'inside_frac'], fontsize=9)
plt.suptitle('Peor (arriba) y mejor (abajo) localizacion - ' + mejor_modelo_nombre)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'gradcam_vs_mascara.png'), dpi=120); plt.show()

## 12. Conclusiones (guion para el minipaper)

Rellena con tus resultados:
- **Validacion cruzada**: combinacion ganadora (lr, dropout, n_bloques) de cada modelo
  y su F1-macro medio ± desviacion tipica sobre los 4 folds.
- **Clasificacion**: comparacion VGG16 vs ResNet50 (accuracy, F1-macro, AUC) y matrices
  de confusion sobre el 20 % de test.
- **Explicabilidad**: valores de IoU/DICE del Grad-CAM frente a la mascara real. Estos
  numeros se comparan en el notebook 3 con los de la U-Net (segmentacion real).
- **Limitaciones**: dataset pequeno y desbalanceado; la mascara es una referencia imperfecta.

La celda siguiente imprime un **resumen con todos los numeros** listos para copiar al
paper. Todas las figuras y CSV se guardan en `resultados/`. El split se guarda en
`resultados/split_indices.npz` para que el notebook 3 use el mismo conjunto de test.

In [ ]:
# ── RESUMEN PARA EL PAPER ─────────────────────────────────────────────────────
# Imprime los numeros clave listos para copiar a paper_BUSI.tex
print('=' * 72)
print('RESUMEN DE RESULTADOS  (copiar a paper_BUSI.tex)')
print('=' * 72)

def _filas_latex_cv(tabla, nombre):
    print('\n%% --- Tabla CV: ' + nombre + ' (ordenada por F1-macro medio) ---')
    print('%%   columnas:  & lr & dropout & n_bloq & F1-macro & epocas \\\\')
    for j, (_, r) in enumerate(tabla.iterrows()):
        f1 = '%.3f$\\pm$%.3f' % (r['f1_macro_mean'], r['f1_macro_std'])
        ep = '%.1f$\\pm$%.1f' % (r['epochs_mean'],   r['epochs_std'])
        if j == 0:
            f1 = '\\textbf{' + f1 + '}'
        print('  & $10^{%d}$ & %.1f & %d & %s & %s \\\\'
              % (int(round(np.log10(r['lr']))), r['dropout'],
                 int(r['n_bloques']), f1, ep))

_filas_latex_cv(cv_vgg, 'VGG16')
_filas_latex_cv(cv_resnet, 'ResNet50')

print('\n%% --- Combinaciones ganadoras (epocas finales = ceil de la media) ---')
for nombre in MODELOS:
    b = mejor[nombre]
    print('%%   %-9s: lr=1e%d  dropout=%.1f  n_bloques=%d  ->  %d epocas finales'
          % (nombre, int(round(np.log10(b['lr']))), b['dropout'],
             int(b['n_bloques']), max(1, int(np.ceil(b['epochs_mean'])))))

print('\n%% --- Tabla de test (accuracy / F1-macro / AUC) ---')
for _, r in comparativa.reset_index().iterrows():
    print('  %-9s & %.3f & %.3f & %.3f \\\\'
          % (r['modelo'], r['accuracy'], r['f1_macro'], r['auc']))

print('\n%% --- Tabla Grad-CAM (IoU / DICE / inside_frac / acierto_pico) ---')
for nombre in MODELOS:
    s = loc[loc['modelo'] == nombre][['IoU', 'DICE', 'inside_frac', 'acierto_pico']].mean()
    print('  %-9s & %.3f & %.3f & %.3f & %.0f\\%% \\\\'
          % (nombre, s['IoU'], s['DICE'], s['inside_frac'], 100 * s['acierto_pico']))

print('\nNota: las metricas de la U-Net (IoU/DICE) se obtienen al ejecutar el notebook 3.')
print('=' * 72)